In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
%pip install -q shap joblib pyarrow xgboost torch

# 03 — SHAP Explanations

For each window pair (A, B), the notebook computes per-replica SHAP attributions on
the flagged set F_{A,B}. The explainer used depends on `MODEL_TYPE`:

| MODEL_TYPE  | Explainer               | Properties                                    |
|-------------|-------------------------|-----------------------------------------------|
| `xgboost`   | `shap.TreeExplainer`    | deterministic, exact; runs on CPU             |
| `logreg`    | — (skipped)             | coefficients are the explanation              |
| `mlp_plr`   | `shap.GradientExplainer`| stochastic, approximate; requires GPU         |

**Output layout** (under `data/shap/{MODEL_TYPE}/pair_{pid:02d}/`):
- `shap_A.npy`, `shap_B.npy` — shape `(R, |F|, p)`, float32
- `expected_values_A.npy`, `expected_values_B.npy` — shape `(R,)`, float32
- `stochasticity.json` — `{best_replica_A, max_abs_diff, median_abs_diff*, is_deterministic}`
  *(`median_abs_diff` only present for MLP-PLR)*
- `global_importance_pair00_A.png` — top-20 global importance plot for pair 0

**Coordinate-system note (MLP-PLR):** MLP-PLR is trained on *scaled* features (each replica
has its own StandardScaler). SHAP attributions are computed in the scaled-feature space and
saved as such. Feature positions still match the column ordering in `feature_names.json` —
only magnitudes differ. Cosine and RBO-on-|attribution| in notebook 04 are robust to this.


In [11]:
import json
import joblib
import numpy as np
import pandas as pd
import shap
from pathlib import Path

WORKSPACE = Path('/content/drive/MyDrive/Homesite_workspace')
PROC_DIR  = WORKSPACE / 'data' / 'processed'
WIN_DIR   = WORKSPACE / 'data' / 'windows'

# ── Model type — must match the MODEL_TYPE used in notebook 02 / 02b ──
MODEL_TYPE = 'logreg'   # 'xgboost' | 'logreg' | 'mlp_plr'

MODEL_DIR = WORKSPACE / 'data' / 'models' / MODEL_TYPE
SHAP_DIR  = WORKSPACE / 'data' / 'shap'  / MODEL_TYPE
SHAP_DIR.mkdir(parents=True, exist_ok=True)

# ── Subsampling (MLP-PLR only) ─────────────────────────────────────────────
# GradientExplainer is the bottleneck of the whole pipeline. We subsample the
# flagged set per pair, mirroring the LIME setup in notebook 03b. The
# subsample positions are saved to mlp_shap_subsample_idx.npy and notebook 04
# uses them automatically (its `_load_attributions` already checks for the
# file). XGBoost SHAP is fast and deterministic, so it stays on the full
# flagged set.
N_MLP_SHAP_SUBSAMPLE = 300    # set to 0 to disable subsampling for MLP-PLR
SEED_BASE            = 42

print(f'SHAP version: {shap.__version__}')
print(f'MODEL_TYPE  : {MODEL_TYPE}')
if MODEL_TYPE == 'mlp_plr':
    print(f'MLP-PLR subsample size: {N_MLP_SHAP_SUBSAMPLE}  (0 = full flagged set)')
MODEL_DIR = WORKSPACE / 'data' / 'models' / MODEL_TYPE
SHAP_DIR  = WORKSPACE / 'data' / 'shap'  / MODEL_TYPE
SHAP_DIR.mkdir(parents=True, exist_ok=True)

print(f'SHAP version: {shap.__version__}')
print(f'MODEL_TYPE  : {MODEL_TYPE}')

SHAP version: 0.51.0
MODEL_TYPE  : logreg
SHAP version: 0.51.0
MODEL_TYPE  : logreg


In [12]:
# Load data and window config
X = pd.read_parquet(PROC_DIR / 'X.parquet').values.astype(np.float32)

with open(PROC_DIR / 'feature_names.json') as f:
    feature_names_json = json.load(f)
feature_names = feature_names_json['all']
num_col_idx   = np.array([feature_names.index(c) for c in feature_names_json['num']], dtype=np.int64)

with open(WIN_DIR / 'window_config.json') as f:
    config = json.load(f)

R     = config['parameters']['R']
pairs = config['pairs']

print(f'X: {X.shape}, features: {len(feature_names)}')
print(f'R={R}, {len(pairs)} pairs')

X: (260753, 317), features: 317
R=3, 5 pairs


## MLP-PLR setup

Loaded only when `MODEL_TYPE == 'mlp_plr'`. Architecture must be byte-identical to notebook 02b.

In [13]:
if MODEL_TYPE == 'mlp_plr':
    import math
    import torch
    import torch.nn as nn
    import torch.nn.functional as F

    assert torch.cuda.is_available(), 'MLP-PLR SHAP requires a GPU runtime.'
    DEVICE = torch.device('cuda')
    print(f'PyTorch : {torch.__version__}')
    print(f'Device  : {torch.cuda.get_device_name(0)}')

    bin_col_idx = np.array([feature_names.index(c) for c in feature_names_json['bin']], dtype=np.int64)

    # ── Architecture (must match notebook 02b exactly) ──
    class PLREmbedding(nn.Module):
        def __init__(self, n_features, k, d_emb, sigma_init):
            super().__init__()
            self.n_features = n_features
            self.k          = k
            self.d_emb      = d_emb
            self.freqs  = nn.Parameter(torch.randn(n_features, k) * sigma_init)
            self.weight = nn.Parameter(torch.empty(n_features, 2 * k, d_emb))
            self.bias   = nn.Parameter(torch.zeros(n_features, d_emb))
            nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))

        def forward(self, x_num):
            scaled = 2 * math.pi * x_num.unsqueeze(-1) * self.freqs
            periodic = torch.cat([torch.sin(scaled), torch.cos(scaled)], dim=-1)
            out = torch.einsum('bpd,pde->bpe', periodic, self.weight) + self.bias
            return F.relu(out).flatten(start_dim=1)

    class MLPPLR(nn.Module):
        def __init__(self, n_num, n_bin, k, d_emb, sigma_init, n_blocks, d_block, dropout):
            super().__init__()
            self.n_num = n_num
            self.n_bin = n_bin
            if n_num > 0:
                self.embedding = PLREmbedding(n_num, k, d_emb, sigma_init)
                input_dim = n_num * d_emb + n_bin
            else:
                self.embedding = None
                input_dim = n_bin
            layers = []
            d_in = input_dim
            for _ in range(n_blocks):
                layers.append(nn.Linear(d_in, d_block))
                layers.append(nn.ReLU())
                layers.append(nn.Dropout(dropout))
                d_in = d_block
            layers.append(nn.Linear(d_in, 1))
            self.backbone = nn.Sequential(*layers)

        def forward(self, x_num, x_bin):
            if self.embedding is not None:
                emb = self.embedding(x_num)
                h = torch.cat([emb, x_bin], dim=1) if self.n_bin > 0 else emb
            else:
                h = x_bin
            return self.backbone(h).squeeze(-1)

    def load_mlp_plr_replica(bundle_path, device):
        bundle = joblib.load(bundle_path)
        m = MLPPLR(**bundle['arch_config'])
        m.load_state_dict(bundle['state_dict'])
        m.to(device).eval()
        return m, bundle['scaler']

    def scale_numeric_inplace(X_raw, scaler, num_idx):
        out = X_raw.astype(np.float32, copy=True)
        out[:, num_idx] = scaler.transform(X_raw[:, num_idx]).astype(np.float32)
        return out

    # Wrapper so GradientExplainer sees a 2-D output (batch, 1).
    # The model takes (x_num, x_bin) separately; we build a single-input wrapper
    # that splits internally so SHAP's API works.
    class ShapInputWrapper(nn.Module):
        def __init__(self, model, num_idx_t, bin_idx_t):
            super().__init__()
            self.model     = model
            self.num_idx_t = num_idx_t
            self.bin_idx_t = bin_idx_t

        def forward(self, x):
            x_num = x.index_select(1, self.num_idx_t)
            x_bin = x.index_select(1, self.bin_idx_t)
            out   = self.model(x_num, x_bin)
            return out.unsqueeze(-1)

    BG_SIZE    = 256
    BATCH_SIZE = 64

    print('MLP-PLR setup complete (classes + helpers defined).')

## SHAP computation loop

One pass per pair. Each branch reads the per-pair models, computes per-replica SHAP on the flagged set, and stores the tensors with the canonical names.

In [14]:
if MODEL_TYPE == 'xgboost':
    import xgboost as xgb

    for p in pairs:
        pid       = p['pair_id']
        pair_dir  = MODEL_DIR / f'pair_{pid:02d}'
        shap_dir  = SHAP_DIR  / f'pair_{pid:02d}'
        shap_dir.mkdir(parents=True, exist_ok=True)

        shap_A_path = shap_dir / 'shap_A.npy'
        shap_B_path = shap_dir / 'shap_B.npy'
        if shap_A_path.exists() and shap_B_path.exists():
            print(f'Pair {pid:02d}: SHAP already computed, skipping.')
            continue

        pred_data = np.load(pair_dir / 'predictions.npz')
        idx_eval  = np.array(p['idx_eval'], dtype=np.int64)
        flagged_local_idx = pred_data['flagged_idx']

        flagged_global_idx = idx_eval[flagged_local_idx]
        X_flagged = X[flagged_global_idx]
        n_flagged = len(X_flagged)
        n_feat    = X_flagged.shape[1]

        print(f'\n── Pair {pid:02d}  ({MODEL_TYPE}) — flagged: {n_flagged:,} ──')

        if n_flagged == 0:
            np.save(shap_A_path, np.zeros((R, 0, n_feat), dtype=np.float32))
            np.save(shap_B_path, np.zeros((R, 0, n_feat), dtype=np.float32))
            np.save(shap_dir / 'expected_values_A.npy', np.zeros(R, dtype=np.float32))
            np.save(shap_dir / 'expected_values_B.npy', np.zeros(R, dtype=np.float32))
            print('  No flagged instances — wrote zeros.')
            continue

        # ── SHAP for replicas of window A ──
        shap_A = np.zeros((R, n_flagged, n_feat), dtype=np.float32)
        ev_A   = np.zeros(R, dtype=np.float32)
        for r in range(R):
            model = joblib.load(pair_dir / 'replicas_A' / f'model_r{r}.joblib')
            explainer = shap.TreeExplainer(model)
            vals = explainer.shap_values(X_flagged)
            if isinstance(vals, list):
                vals = vals[1]  # positive class
            shap_A[r] = vals.astype(np.float32)
            raw_ev = explainer.expected_value
            ev_A[r] = np.float32(raw_ev if not isinstance(raw_ev, list) else raw_ev[1])
            print(f'  A replica {r}: |mean|={np.abs(shap_A[r]).mean():.5f}  base={ev_A[r]:.4f}')

        # ── SHAP for replicas of window B ──
        shap_B = np.zeros((R, n_flagged, n_feat), dtype=np.float32)
        ev_B   = np.zeros(R, dtype=np.float32)
        for r in range(R):
            model = joblib.load(pair_dir / 'replicas_B' / f'model_r{r}.joblib')
            explainer = shap.TreeExplainer(model)
            vals = explainer.shap_values(X_flagged)
            if isinstance(vals, list):
                vals = vals[1]
            shap_B[r] = vals.astype(np.float32)
            raw_ev = explainer.expected_value
            ev_B[r] = np.float32(raw_ev if not isinstance(raw_ev, list) else raw_ev[1])
            print(f'  B replica {r}: |mean|={np.abs(shap_B[r]).mean():.5f}  base={ev_B[r]:.4f}')

        # ── Sanity: SHAP values should sum to log-odds prediction − base_value ──
        model_A0  = joblib.load(pair_dir / 'replicas_A' / 'model_r0.joblib')
        preds_raw = model_A0.predict(xgb.DMatrix(X_flagged), output_margin=True)
        shap_sum  = shap_A[0].sum(axis=1) + ev_A[0]
        max_err   = np.abs(shap_sum - preds_raw).max()
        print(f'  Sanity (A r0): max |SHAP_sum - log_odds| = {max_err:.6f}')

        np.save(shap_A_path, shap_A)
        np.save(shap_B_path, shap_B)
        np.save(shap_dir / 'expected_values_A.npy', ev_A)
        np.save(shap_dir / 'expected_values_B.npy', ev_B)

elif MODEL_TYPE == 'mlp_plr':
    num_idx_t = torch.from_numpy(num_col_idx).long().to(DEVICE)
    bin_idx_t = torch.from_numpy(bin_col_idx).long().to(DEVICE)

    for p in pairs:
        pid       = p['pair_id']
        pair_dir  = MODEL_DIR / f'pair_{pid:02d}'
        shap_dir  = SHAP_DIR  / f'pair_{pid:02d}'
        shap_dir.mkdir(parents=True, exist_ok=True)

        shap_A_path = shap_dir / 'shap_A.npy'
        shap_B_path = shap_dir / 'shap_B.npy'
        if shap_A_path.exists() and shap_B_path.exists():
            print(f'Pair {pid:02d}: SHAP already computed, skipping.')
            continue

        pred_data         = np.load(pair_dir / 'predictions.npz')
        idx_A             = np.array(p['idx_A'], dtype=np.int64)
        idx_B             = np.array(p['idx_B'], dtype=np.int64)
        idx_eval          = np.array(p['idx_eval'], dtype=np.int64)
        flagged_local_idx = pred_data['flagged_idx']
        n_flagged_total   = len(flagged_local_idx)

        flagged_global_idx = idx_eval[flagged_local_idx]
        X_flagged_full     = X[flagged_global_idx]
        n_feat             = X_flagged_full.shape[1]

        # ── Subsample positions within flagged_idx (mirrors LIME in notebook 03b) ──
        # The saved indices are positions inside flagged_local_idx, NOT global row ids.
        # Notebook 04 detects mlp_shap_subsample_idx.npy and uses it as the subset.
        sub_path = shap_dir / 'mlp_shap_subsample_idx.npy'
        if N_MLP_SHAP_SUBSAMPLE > 0 and n_flagged_total > N_MLP_SHAP_SUBSAMPLE:
            rng_sub = np.random.default_rng(SEED_BASE + pid * 100)
            mlp_sub = np.sort(
                rng_sub.choice(n_flagged_total, size=N_MLP_SHAP_SUBSAMPLE, replace=False)
            ).astype(np.int64)
        else:
            mlp_sub = np.arange(n_flagged_total, dtype=np.int64)
        np.save(sub_path, mlp_sub)

        X_flagged = X_flagged_full[mlp_sub]
        n_flagged = len(X_flagged)

        print(f'\n── Pair {pid:02d}  ({MODEL_TYPE}) — flagged: {n_flagged_total:,}  '
              f'SHAP subsample: {n_flagged} ──')

        if n_flagged == 0:
            np.save(shap_A_path, np.zeros((R, 0, n_feat), dtype=np.float32))
            np.save(shap_B_path, np.zeros((R, 0, n_feat), dtype=np.float32))
            np.save(shap_dir / 'expected_values_A.npy', np.zeros(R, dtype=np.float32))
            np.save(shap_dir / 'expected_values_B.npy', np.zeros(R, dtype=np.float32))
            continue

        shap_A = np.zeros((R, n_flagged, n_feat), dtype=np.float32)
        ev_A   = np.zeros(R, dtype=np.float32)
        shap_B = np.zeros((R, n_flagged, n_feat), dtype=np.float32)
        ev_B   = np.zeros(R, dtype=np.float32)

        for AB, idx_window, shap_out, ev_out in [
            ('A', idx_A, shap_A, ev_A),
            ('B', idx_B, shap_B, ev_B),
        ]:
            for r in range(R):
                bundle_path = pair_dir / f'replicas_{AB}' / f'model_r{r}.joblib'
                model, scaler = load_mlp_plr_replica(bundle_path, DEVICE)

                # Background dataset from the replica's training window
                rng_bg = np.random.default_rng(r)
                n_bg   = min(BG_SIZE, len(idx_window))
                idx_bg = rng_bg.choice(idx_window, size=n_bg, replace=False)
                X_bg   = scale_numeric_inplace(X[idx_bg], scaler, num_col_idx)
                bg_t   = torch.from_numpy(X_bg).float().to(DEVICE)

                X_fl_s = scale_numeric_inplace(X_flagged, scaler, num_col_idx)
                fl_t   = torch.from_numpy(X_fl_s).float().to(DEVICE)

                wrapper   = ShapInputWrapper(model, num_idx_t, bin_idx_t).to(DEVICE).eval()
                explainer = shap.GradientExplainer(wrapper, bg_t)

                shap_batches = []
                for start in range(0, n_flagged, BATCH_SIZE):
                    chunk = fl_t[start:start + BATCH_SIZE]
                    vals  = explainer.shap_values(chunk)
                    if isinstance(vals, list):
                        vals = vals[0]
                    if isinstance(vals, np.ndarray) and vals.ndim == 3 and vals.shape[-1] == 1:
                        vals = vals[..., 0]
                    shap_batches.append(vals)
                shap_out[r] = np.concatenate(shap_batches, axis=0).astype(np.float32)

                # Base value: mean logit over background
                with torch.no_grad():
                    ev_out[r] = float(wrapper(bg_t).mean().item())

                print(f'  {AB} replica {r}: |mean|={np.abs(shap_out[r]).mean():.5f}  base={ev_out[r]:.4f}')

                del model, wrapper, explainer, bg_t, fl_t
                torch.cuda.empty_cache()

        np.save(shap_A_path, shap_A)
        np.save(shap_B_path, shap_B)
        np.save(shap_dir / 'expected_values_A.npy', ev_A)
        np.save(shap_dir / 'expected_values_B.npy', ev_B)

elif MODEL_TYPE == 'logreg':
    print(f'MODEL_TYPE is "{MODEL_TYPE}" — SHAP computation skipped.')
    print('Logistic Regression uses its coefficients as the explanation; no post-hoc XAI is needed.')

else:
    raise ValueError(f'Unknown MODEL_TYPE: {MODEL_TYPE}')

print('\n✓ SHAP computation done.')

MODEL_TYPE is "logreg" — SHAP computation skipped.
Logistic Regression uses its coefficients as the explanation; no post-hoc XAI is needed.

✓ SHAP computation done.


## SHAP stochasticity diagnostic

TreeSHAP is deterministic — running the explainer twice on the same model and data returns identical values.
GradientExplainer is mildly stochastic (depends on background sampling). We verify per pair using the best A
replica (highest PR-AUC) and save `stochasticity.json`.

In [15]:
if MODEL_TYPE == 'xgboost':
    for p in pairs:
        pid      = p['pair_id']
        pair_dir = MODEL_DIR / f'pair_{pid:02d}'
        shap_dir = SHAP_DIR  / f'pair_{pid:02d}'
        out_path = shap_dir / 'stochasticity.json'
        if out_path.exists():
            continue

        pred_data = np.load(pair_dir / 'predictions.npz')
        flagged_local_idx = pred_data['flagged_idx']
        idx_eval = np.array(p['idx_eval'], dtype=np.int64)

        if len(flagged_local_idx) == 0:
            json_payload = {'best_replica_A': 0, 'max_abs_diff': 0.0, 'is_deterministic': True}
            with open(out_path, 'w') as f:
                json.dump(json_payload, f, indent=2)
            continue

        per_rep = pred_data['per_replica_pr_auc_A']
        best_r  = int(np.argmax(per_rep))

        X_flagged = X[idx_eval[flagged_local_idx]]
        model = joblib.load(pair_dir / 'replicas_A' / f'model_r{best_r}.joblib')

        ex1 = shap.TreeExplainer(model)
        ex2 = shap.TreeExplainer(model)
        v1  = ex1.shap_values(X_flagged)
        v2  = ex2.shap_values(X_flagged)
        if isinstance(v1, list):
            v1 = v1[1]
            v2 = v2[1]
        max_abs_diff = float(np.abs(v1 - v2).max())

        with open(out_path, 'w') as f:
            json.dump({
                'best_replica_A':   best_r,
                'max_abs_diff':     max_abs_diff,
                'is_deterministic': bool(max_abs_diff < 1e-8),
            }, f, indent=2)
        print(f'Pair {pid:02d}: best_r={best_r}  max_abs_diff={max_abs_diff:.2e}')

elif MODEL_TYPE == 'mlp_plr':
    num_idx_t = torch.from_numpy(num_col_idx).long().to(DEVICE)
    bin_idx_t = torch.from_numpy(bin_col_idx).long().to(DEVICE)

    for p in pairs:
        pid      = p['pair_id']
        pair_dir = MODEL_DIR / f'pair_{pid:02d}'
        shap_dir = SHAP_DIR  / f'pair_{pid:02d}'
        out_path = shap_dir / 'stochasticity.json'
        if out_path.exists():
            continue

        pred_data = np.load(pair_dir / 'predictions.npz')
        flagged_local_idx = pred_data['flagged_idx']
        idx_A    = np.array(p['idx_A'],    dtype=np.int64)
        idx_eval = np.array(p['idx_eval'], dtype=np.int64)

        if len(flagged_local_idx) == 0:
            with open(out_path, 'w') as f:
                json.dump({'best_replica_A': 0, 'max_abs_diff': 0.0,
                           'median_abs_diff': 0.0, 'is_deterministic': True}, f, indent=2)
            continue

        per_rep = pred_data['per_replica_pr_auc_A']
        best_r  = int(np.argmax(per_rep))

        # Use the same subsample as the main loop, so stochasticity is measured
        # on the exact instances that ended up in shap_A.npy.
        sub_path = shap_dir / 'mlp_shap_subsample_idx.npy'
        if sub_path.exists():
            mlp_sub = np.load(sub_path)
        else:
            mlp_sub = np.arange(len(flagged_local_idx), dtype=np.int64)
        X_flagged = X[idx_eval[flagged_local_idx]][mlp_sub]

        bundle_path = pair_dir / 'replicas_A' / f'model_r{best_r}.joblib'
        model, scaler = load_mlp_plr_replica(bundle_path, DEVICE)
        wrapper = ShapInputWrapper(model, num_idx_t, bin_idx_t).to(DEVICE).eval()

        diffs = []
        for seed in (0, 1):
            rng = np.random.default_rng(seed)
            n_bg = min(BG_SIZE, len(idx_A))
            idx_bg = rng.choice(idx_A, size=n_bg, replace=False)
            X_bg   = scale_numeric_inplace(X[idx_bg], scaler, num_col_idx)
            bg_t   = torch.from_numpy(X_bg).float().to(DEVICE)

            X_fl_s = scale_numeric_inplace(X_flagged, scaler, num_col_idx)
            fl_t   = torch.from_numpy(X_fl_s).float().to(DEVICE)

            ex = shap.GradientExplainer(wrapper, bg_t)
            v  = ex.shap_values(fl_t)
            if isinstance(v, list):
                v = v[0]
            if isinstance(v, np.ndarray) and v.ndim == 3 and v.shape[-1] == 1:
                v = v[..., 0]
            diffs.append(v.astype(np.float32))

        max_abs_diff    = float(np.abs(diffs[0] - diffs[1]).max())
        median_abs_diff = float(np.median(np.abs(diffs[0] - diffs[1])))

        with open(out_path, 'w') as f:
            json.dump({
                'best_replica_A':   best_r,
                'max_abs_diff':     max_abs_diff,
                'median_abs_diff':  median_abs_diff,
                'is_deterministic': bool(max_abs_diff < 1e-6),
            }, f, indent=2)
        print(f'Pair {pid:02d}: best_r={best_r}  max_abs_diff={max_abs_diff:.4f}  median_abs_diff={median_abs_diff:.4f}')

        del model, wrapper
        torch.cuda.empty_cache()

else:
    print(f'MODEL_TYPE={MODEL_TYPE} — no SHAP stochasticity to measure.')

MODEL_TYPE=logreg — no SHAP stochasticity to measure.


## Quick SHAP summary (pair 0)

Top-20 global SHAP importances for the first pair, window A.

In [16]:
if MODEL_TYPE in ('xgboost', 'mlp_plr'):
    import matplotlib.pyplot as plt

    shap_A_0 = np.load(SHAP_DIR / 'pair_00' / 'shap_A.npy')   # (R, |F|, p)
    phi_bar_A  = shap_A_0.mean(axis=0)                         # (|F|, p)
    global_imp = np.abs(phi_bar_A).mean(axis=0)

    top_k   = min(20, len(global_imp))
    top_idx = np.argsort(global_imp)[::-1][:top_k]

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh(
        [feature_names[i] for i in top_idx[::-1]],
        global_imp[top_idx[::-1]],
        color='steelblue',
    )
    ax.set_title(f'Top {top_k} global SHAP importances (pair 0, model A, {MODEL_TYPE})')
    ax.set_xlabel('Mean |SHAP value|')
    plt.tight_layout()
    plt.savefig(SHAP_DIR / 'global_importance_pair00_A.png', dpi=120)
    plt.show()
else:
    print(f'No SHAP plot for MODEL_TYPE={MODEL_TYPE}.')

No SHAP plot for MODEL_TYPE=logreg.
